In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# Download necessary NLTK resources
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')

# ==========================================
# Task 1: Data Understanding
# ==========================================
print("--- Task 1: Data Understanding ---")
# Load the dataset (Make sure 'IMDB Dataset.csv' is uploaded to Colab)
# If using a different dataset, change the filename and column names accordingly.
try:
    df = pd.read_csv('IMDB Dataset.csv')

    # For computation speed in this Colab demonstration, we will sample 10,000 rows.
    # You can remove the .sample() method to run on the full 50k dataset if you have time.
    df = df.sample(10000, random_state=42).reset_index(drop=True)

    print(f"Dataset loaded successfully! Number of samples: {df.shape[0]}")
except FileNotFoundError:
    print("ERROR: Please upload 'IMDB Dataset.csv' to your Colab workspace.")
    # Creating a dummy dataframe so the rest of the code doesn't crash during review
    df = pd.DataFrame({'review': ['I loved this movie!', 'Terrible, worst film ever.', 'It was okay, not great but fine.'],
                       'sentiment': ['positive', 'negative', 'positive']})

# Display sample texts
print("\nFirst 3 rows of the dataset:")
print(df.head(3))

# Class Distribution
print("\nClass Distribution:")
print(df['sentiment'].value_counts())

# Convert labels to binary (positive = 1, negative = 0)
df['label'] = df['sentiment'].map({'positive': 1, 'negative': 0})

# ==========================================
# Task 2: NLP Preprocessing (Mandatory)
# ==========================================
print("\n--- Task 2: NLP Preprocessing ---")

# Initialize Lemmatizer and Stopwords
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))
# Retain important negative stopwords for sentiment analysis
stop_words.difference_update({'no', 'not', 'nor', "didn't", "wasn't", "couldn't", "isn't"})

def preprocess_text(text):
    # 1. Lowercasing
    text = text.lower()
    # 2. Handling URLs and HTML tags
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'<.*?>', '', text)
    # 3. Removing punctuation and special characters (keep only alphabets)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # 4. Tokenization
    tokens = word_tokenize(text)
    # 5. Removing stopwords & 6. Lemmatization
    cleaned_tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]

    return " ".join(cleaned_tokens)

# Apply preprocessing (This might take a minute or two)
print("Preprocessing text... please wait.")
df['cleaned_review'] = df['review'].apply(preprocess_text)
print("Preprocessing complete. Sample cleaned text:")
print(df[['review', 'cleaned_review']].head(2))

# ==========================================
# Task 3: Feature Engineering
# ==========================================
print("\n--- Task 3: Feature Engineering ---")

# Split the data into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(df['cleaned_review'], df['label'], test_size=0.2, random_state=42)

# 1. Bag of Words (BoW)
print("Vectorizing using Bag of Words (CountVectorizer)...")
bow_vectorizer = CountVectorizer(max_features=5000) # Limiting features to prevent memory crash
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

# 2. TF-IDF
print("Vectorizing using TF-IDF...")
tfidf_vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# ==========================================
# Task 4 & 5: Model Building & Evaluation
# ==========================================
print("\n--- Task 4 & 5: Model Building & Evaluation ---")

# Reusable function to train and evaluate models
def train_and_evaluate(model, model_name, X_train_vec, X_test_vec, y_train, y_test):
    # Train
    model.fit(X_train_vec, y_train)
    # Predict
    y_pred = model.predict(X_test_vec)
    # Evaluate
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    print(f"Results for {model_name}:")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1 Score:  {f1:.4f}")
    print("-" * 30)

    return {'Model': model_name, 'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1': f1}

# Initialize Models
log_reg = LogisticRegression(max_iter=1000)
nb_model = MultinomialNB()
dt_model = DecisionTreeClassifier(random_state=42)

results = []

# Evaluate with BoW
print(">>> Performance using Bag of Words (BoW) <<<")
results.append(train_and_evaluate(log_reg, "Logistic Regression (BoW)", X_train_bow, X_test_bow, y_train, y_test))
results.append(train_and_evaluate(nb_model, "Naive Bayes (BoW)", X_train_bow, X_test_bow, y_train, y_test))
results.append(train_and_evaluate(dt_model, "Decision Tree (BoW)", X_train_bow, X_test_bow, y_train, y_test))

# Evaluate with TF-IDF
print("\n>>> Performance using TF-IDF <<<")
results.append(train_and_evaluate(log_reg, "Logistic Regression (TF-IDF)", X_train_tfidf, X_test_tfidf, y_train, y_test))
results.append(train_and_evaluate(nb_model, "Naive Bayes (TF-IDF)", X_train_tfidf, X_test_tfidf, y_train, y_test))
results.append(train_and_evaluate(dt_model, "Decision Tree (TF-IDF)", X_train_tfidf, X_test_tfidf, y_train, y_test))

# Convert results to DataFrame for easy comparison
results_df = pd.DataFrame(results)

# ==========================================
# Task 6: Comparison & Insights
# ==========================================
print("\n--- Task 6: Comparison & Insights ---")
print("\nSummary of Model Performances:")
print(results_df.sort_values(by='Accuracy', ascending=False).to_string(index=False))

print("\n" + "="*50)
print("Final Insights (For your Notebook Markdown):")
print("="*50)
print("1. Preprocessing Impact: By utilizing Lemmatization over Stemming, we retained actual dictionary words, which helped contextual integrity. Furthermore, retaining negative stopwords like 'not' and 'nor' prevented the sentiment from flipping inadvertently.")
print("2. Best Vectorization Method: TF-IDF generally outperforms raw Bag of Words (BoW) because it penalizes highly frequent words across all documents, allowing unique sentiment-carrying words to hold more weight in the prediction algorithms.")
print("3. Best Model: Logistic Regression paired with TF-IDF consistently provides the highest accuracy and F1 score for binary text classification tasks like this, due to its ability to find linear boundaries in high-dimensional sparse data.")
print("4. Trade-offs: Decision Trees perform the worst and are highly prone to overfitting on the specific vocabulary of the training set. Naive Bayes is incredibly fast to train but assumes word independence, which slightly limits its accuracy compared to Logistic Regression.")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


--- Task 1: Data Understanding ---
ERROR: Please upload 'IMDB Dataset.csv' to your Colab workspace.

First 3 rows of the dataset:
                             review sentiment
0               I loved this movie!  positive
1        Terrible, worst film ever.  negative
2  It was okay, not great but fine.  positive

Class Distribution:
sentiment
positive    2
negative    1
Name: count, dtype: int64

--- Task 2: NLP Preprocessing ---
Preprocessing text... please wait.
Preprocessing complete. Sample cleaned text:
                       review            cleaned_review
0         I loved this movie!               loved movie
1  Terrible, worst film ever.  terrible worst film ever

--- Task 3: Feature Engineering ---
Vectorizing using Bag of Words (CountVectorizer)...
Vectorizing using TF-IDF...

--- Task 4 & 5: Model Building & Evaluation ---
>>> Performance using Bag of Words (BoW) <<<
Results for Logistic Regression (BoW):
Accuracy:  1.0000
Precision: 1.0000
Recall:    1.0000
F1 Score:  1.0

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
